In [ ]:
%pip install tabpfn
%pip install tabpfn-client
%pip install matplotlib
%matplotlib inline

import pandas as pd
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNClassifier
from sklearn.metrics import accuracy_score

# Datensatz laden und bereinigen


In [ ]:
# 1. Datensatz aus Github-URL als CSV einlesen
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
data = pd.read_csv(url)

data.head()


# 2.1 TotalCharges von Text in echte Zahlen umwandeln
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce').fillna(0)

# 2.2 customerID löschen (das ist nur ein Name, kein Merkmal fürs Modell)
data = data.drop(columns=['customerID'])

# 2.3 Zielvariable Churn (Yes/No) in 1 und 0 umwandeln
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})


# 3. Features und Zielvariable trennen
X = data.drop(columns=['Churn'])
y = data['Churn']


display(data)

### 🛠️ Backend-Auswahl: Lokale Ausführung vs. Cloud-API

TabPFN bietet zwei Betriebsmodi für flexible Einsatzszenarien:

1. **Lokaler Modus (Standard & Datenschutz-Fokus):**
   * **Vorteil:** Alle Daten verbleiben zu 100 % lokal auf dem eigenen Rechner (DSGVO-konform, offline nutzbar).
   * **Ideal für:** Sensible Unternehmensdaten (wie Kundendaten, Finanzwerte) im echten Produktionseinsatz.

2. **Cloud-API Modus (`tabpfn_client`):**
   * **Vorteil:** Verlagert die Berechnung auf die Server von Prior Labs.
   * **Ideal für:** Schwächere Laptops oder schnelle Testumgebungen ohne dedizierte Hardware.

> **Hinweis zur Einrichtung der Cloud-API:**
> Für die Cloud-Nutzung ist ein kostenloser Account bei Prior Labs erforderlich. Erstelle deinen Token unter [ux.priorlabs.ai/account](https://ux.priorlabs.ai/account) und trage ihn unten im Code ein:
> ```python
> from tabpfn_client import set_access_token
> set_access_token("DEIN_TOKEN_HIER_EINFUEGEN")
> ```

HIER: **Betriebsart auswählen**

In [ ]:
# True  = Über die Cloud-API (schont schwache Laptops, erfordert einen Prior Labs Token)
# False = Lokal auf dem Computer (vollständig datenschutzkonform, läuft komplett offline)
USE_CLOUD_API = False  

In [ ]:
if USE_CLOUD_API:
    try:
        from tabpfn_client import TabPFNClassifier, init, set_access_token
        
        # Optional: Trage hier deinen Token ein, falls er nicht als Umgebungsvariable gesetzt ist
        set_access_token("DEIN_TOKEN_HIER_EINFUEGEN")
        
        init()
        classifier = TabPFNClassifier()
        print("✅ Modus: Cloud-API aktiv (Berechnung auf Prior Labs Servern).")
    except ImportError:
        print("❌ Fehler: 'tabpfn_client' ist nicht installiert. Bitte führe '!pip install tabpfn-client' aus.")
else:
    # Lokale Variante (Datenschutz-Fokus für KMUs)
    from tabpfn import TabPFNClassifier
    classifier = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)
    print("✅ Modus: Lokale Ausführung aktiv")

# Modelltraining mit TabPFN

In [ ]:
# 1. Train-Test-Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. TabPFN initialisieren (läuft lokal auf der CPU)
classifier = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)

print("Modell wird trainiert...")
classifier.fit(X_train, y_train)

#clear_output(wait=True)

# 3. Vorhersage und Genauigkeit berechnen
y_pred_tabpfn = classifier.predict(X_test)

acc_tabpfn = accuracy_score(y_test, y_pred_tabpfn)
print("Genauigkeit (Accuracy) von TabPFN:", acc_tabpfn)

# BENCHMARK: Vergleich mit Random Forest
**Warum Random Forest?** 
    Er gilt im Data Science als einer der robustesten und beliebtesten Standard-Klassifikatoren für tabellarische Daten. Er liefert solide Ergebnisse, ohne dass man stundenlang an Parametern schrauben muss – genau wie TabPFN. Das macht ihn zum idealen Vergleichspartner.


- **Die technische Hürde (Datenaufbereitung):** 
    Ein klassischer Random Forest kann rein mathematisch **nur mit Zahlen** rechnen. Textspalten wie `"Male"` / `"Female"` oder Verträge können von dem Algorithmus nicht direkt verarbeitet werden.
- **Die Lösung (Pipeline & ColumnTransformer):** 
    Mithilfe einer `Pipeline` und eines `ColumnTransformer` automatisieren wir diesen Schritt: Textspalten werden automatisch erkannt und per `OrdinalEncoder` in Zahlen übersetzt, während reine Zahlen-Spalten einfach unverändert durchgelassen werden. Das hält den Workflow modular und schützt vor Datenlecks.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 1. Alle Spalten mit kategorischen Daten identifizieren
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Den ColumnTransformer einrichten
column_transformer = ColumnTransformer(
    transformers=[
        ('CatToNum', OrdinalEncoder(), categorical_cols)
    ],
    remainder='passthrough'  # Numerische Spalten einfach unverändert durchlassen
)

# 3. Pipeline erstellen: Erst umwandeln, dann Random Forest trainieren
rf_pipeline = make_pipeline(column_transformer, RandomForestClassifier(random_state=42))

print("Random Forest Pipeline wird trainiert...")
rf_pipeline.fit(X_train, y_train)

# 4. Vorhersage ausführen
y_pred_rf = rf_pipeline.predict(X_test)

# clear_output(wait=True) 

acc_rf = accuracy_score(y_test, y_pred_rf)
print("Genauigkeit von Random Forest:", acc_rf)

# direkter Vergleich

In [ ]:
# Direkter Leistungsvergleich
print("--- MODELLVERGLEICH ---")
print(f"Random Forest Genauigkeit: {acc_rf:.4f}")
print(f"TabPFN Genauigkeit: {acc_tabpfn:.4f}")

# 5. Visueller Vergleich als Balkendiagramm
results_df = pd.DataFrame({
    'Model': ['Random Forest', 'TabPFN'],
    'Accuracy': [acc_rf, acc_tabpfn]
})

ax = results_df.plot(x='Model', y='Accuracy', kind='bar', ylim=(0.7, 0.9), figsize=(8, 4), legend=False, color=['skyblue', 'salmon'])
ax.set_title("Modellvergleich: Einfacher Train-Test-Split")
ax.set_ylabel("Genauigkeit (Accuracy)")

- **Ergebnisse:** 
    Während der Random Forest ein stabiles, bewährtes Arbeitstier ist, zeigt der direkte Vergleich, wie sich moderne Ansätze (wie TabPFN) schlagen. Oft liefert TabPFN direkt out-of-the-box extrem starke Ergebnisse, ohne dass man sich um komplexe Hyperparameter-Suche (Grid Search) kümmern muss.
    
- **Fazit für die Praxis:** 
    Für KMUs ist oft nicht nur die reine Genauigkeit entscheidend, sondern auch die **Zeit** und der **Aufwand**, die man in die Datenaufbereitung und das Tuning stecken muss. Pipelines helfen uns dabei, diesen Prozess reproduzierbar und fehlerfrei zu halten.

---
## 🛠️ Profi-Ausblick: Automatisierter Vergleich mit Kreuzvalidierung
In der realen Data-Science-Praxis verlässt man sich ungern auf *einen* zufälligen Train-Test-Split (vielleicht erwischt man zufällig einen "leichten" oder "schweren" Testdatensatz). 

Profis nutzen deshalb eine **Kreuzvalidierung (Cross-Validation)**: Das Modell wird mehrfach auf unterschiedlichen Teilen der Daten trainiert und die Leistung gemittelt. Hier siehst du, wie man mehrere Algorithmen (Random Forest, XGBoost, CatBoost und TabPFN) in einer eleganten Schleife gegeneinander antreten lässt:

In [ ]:
# 1. Benötigte Bibliotheken für XGBoost und CatBoost installieren (falls noch nicht geschehen)
%pip install xgboost catboost

# 2. Imports für die zusätzlichen Modelle und die Kreuzvalidierung
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ziel-Label sicherheitshalber für numerische Metriken encodieren
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Mehrere Modelle in einer Liste definieren
models = [
    ("TabPFN", classifier),  # Dein initialisiertes TabPFN-Modell
    (
        "RandomForest",
        make_pipeline(
            column_transformer,  
            RandomForestClassifier(random_state=42),
        ),
    ),
    (
        "XGBoost",
        make_pipeline(
            column_transformer,  
            XGBClassifier(random_state=42),
        ),
    ),
    (
        "CatBoost",
        make_pipeline(
            column_transformer,  
            CatBoostClassifier(random_state=42, verbose=0),
        ),
    ),
]

# Kreuzvalidierung einrichten (Stratified sorgt für ausgeglichene Klassen)
n_splits = 3
cv = StratifiedKFold(n_splits=n_splits, random_state=42, shuffle=True)
scoring = "roc_auc_ovr" if len(np.unique(y_encoded)) > 2 else "roc_auc"

print("⏳ Starte automatisierte Kreuzvalidierung über alle Modelle...")
scores = {
    name: cross_val_score(model, X, y_encoded, cv=cv, scoring=scoring, n_jobs=1, verbose=1)
    for name, model in models
}

# Ergebnisse als DataFrame aufbereiten und plotten
df_cv = pd.DataFrame(
    [(k, v.mean()) for (k, v) in scores.items()], columns=["Model", "ROC AUC"]
)

ax = df_cv.plot(x="Model", y="ROC AUC", kind="bar", figsize=(10, 5), color='mediumseagreen')
ax.set_ylim(df_cv["ROC AUC"].min() * 0.99, min(1.0, df_cv["ROC AUC"].max() * 1.01))
ax.set_title(f"Profi-Modellvergleich - {n_splits}-fold Cross-Validation (ROC AUC)")
ax.set_ylabel("ROC AUC Score")
plt.show()